# Executor de experimentos

- Usado para a execução de cada experimento no ambiente do Google Colab.
- Cada experimento foi realizado em três rodadas com seeds diferentes
- Os resultados de cada experimento foram salvas em arquivos no Google Drive e num banco de dados

Verificação do ambiente de execução

In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

!nvidia-smi

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Wed May  6 21:14:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                       

Clonagem do repositório atualizado no GitHub

In [2]:
%cd /content
!git clone https://github.com/amartinsmg/classification-of-medical-images-using-cnn.git
%cd /content/classification-of-medical-images-using-cnn

/content
Cloning into 'classification-of-medical-images-using-cnn'...
remote: Enumerating objects: 797, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 797 (delta 84), reused 147 (delta 62), pack-reused 612 (from 1)
Receiving objects: 100% (797/797), 7.57 MiB | 13.42 MiB/s, done.
Resolving deltas: 100% (410/410), done.
/content/classification-of-medical-images-using-cnn


Montagem do Google Drive

In [3]:
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/classification-of-medical-images-using-cnn/"

Mounted at /content/drive


Configuração dos parâmetros do experimento

In [4]:
EXPERIMENT_NAME = "efficientnet-rescaling"
BASE_MODEL = "efficientnet"
NORMALIZATION = "rescaling"
DATA_AUG = True
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_WEIGHT = False
LEARNING_RATE = 0.001
EPOCHS = 10
THRESHOLD = 0.5
seeds = [42, 123, 999]

Criação da engine do banco de dados
- Foi usado um banco de dados sqlite hospedado também no Google Drive

In [5]:
from src.db import insert_run, get_engine

DB_PATH = f"sqlite:///{BASE_PATH}/experiments.db"

engine = get_engine(DB_PATH)

Realização das três rodadas do experimento com:
- Treinamento do modelo
- Teste do modelo treinado
- Persitência dos parâmetros e dos resulados da rodada em banco de dados

In [6]:
from src.train import train_pipeline
from src.test import test_pipeline

for i in range(3):
    run_id = i + 1
    config = train_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        data_augmentation=DATA_AUG,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_weights=CLASS_WEIGHT,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        seed=seeds[i],
    )

    metrics = test_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        threshold=THRESHOLD,
    )

    insert_run(
        engine=engine,
        experiment=EXPERIMENT_NAME,
        run_name=f"run-{run_id}",
        config=config,
        metrics=metrics,
    )

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 1039s 6s/step - AUC: 0.5094 - accuracy: 0.7425 - loss: 0.5789 - val_AUC: 0.5000 - val_accuracy: 0.5000 - val_loss: 1.0337
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - AUC: 0.5036 - accuracy: 0.7429 - loss: 0.5759 - val_AUC: 0.6250 - val_accuracy: 0.5000 - val_loss: 1.0285
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 34ms/step - AUC: 0.5063 - accuracy: 0.7429 - loss: 0.5750 - val_AUC: 0.5000 - val_accuracy: 0.5000 - val_loss: 1.0392
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - AUC: 0.4983 - accuracy: 0.7429 - loss: 0.5757 - val_AUC: 0.5000 - val_accuracy: 0.5000 - val_loss: 0.9655
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - AUC: 0.4955 - accuracy: 0.7429 - loss: 0.5740 - val_AUC: 0.5625 - val_accuracy: 0.5000 - val_loss: 0.9160
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - AUC: 0.

In [7]:
!pip install -q dvc dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.

In [8]:
import dagshub
from google.colab import userdata

dagshub.auth.add_app_token(token=userdata.get("DAGSHUB_TOKEN"))

In [9]:
RESULTS_RELATIVE_PATH = "results/" + EXPERIMENT_NAME
LOCAL_PATH = BASE_PATH + "/" + RESULTS_RELATIVE_PATH

dagshub.upload_files(
    "amartinsmg/classification-of-medical-images-using-cnn",
    local_path=LOCAL_PATH,
    remote_path=RESULTS_RELATIVE_PATH,
)

Accessing as amartinsmg

Output()

Directory upload complete, uploaded 9 files to 
https://dagshub.com/amartinsmg/classification-of-medical-images-using-cnn/src/main/results%2Fefficientnet-rescaling